# Steps 4–5: Results Routing & Clinical Follow-Up Pathways

This notebook covers what happens **after** a screening result is returned:

- **Step 4 — Result routing**
  - Cervical: NORMAL → surveillance; abnormal → colposcopy referral (with LTFU check);
    HPV+/normal cyto → 1-year repeat or colposcopy.
  - Other cancers: NEGATIVE → surveillance; POSITIVE → stub referral pathway.

- **Step 5 — Follow-up execution**
  - Colposcopy → CIN grade draw → LEEP / cone / surveillance.
  - Loss-to-follow-up modelled explicitly at two nodes.

All logic lives in `followup.py`. This notebook demonstrates and tests it.

In [1]:
import sys, random
sys.path.insert(0, '.')

import config as cfg
from patient import Patient
from population import sample_patient
from followup import (
    route_cervical_result,
    run_colposcopy,
    run_treatment,
    run_cervical_followup,
    run_stub_followup,
)
from metrics import initialize_metrics

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

Imports OK


## Result Routing — Cervical
Show which pathway each result category triggers.

In [2]:
results_to_test = [
    'NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POS_NORMAL_CYTO'
]

# Run 200 trials per result to show routing probabilities (LTFU is stochastic)
from collections import Counter

print(f'{"Result":<25}  Routing distribution (200 trials)')
print('-' * 70)
for result in results_to_test:
    routes = []
    for _ in range(200):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, result
        routes.append(route_cervical_result(p, day := 0))
    cnt = Counter(routes)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {result:<23}  {summary}')

Result                     Routing distribution (200 trials)
----------------------------------------------------------------------
  NORMAL                   routine_surveillance=200
  ASCUS                    colposcopy=167  exit=33
  LSIL                     colposcopy=164  exit=36
  ASC-H                    colposcopy=165  exit=35
  HSIL                     colposcopy=153  exit=47
  HPV_POS_NORMAL_CYTO      colposcopy=103  exit=32  one_year_repeat=65


## Colposcopy Result Distribution
Verify CIN grade draws match config probabilities for each triggering result.

In [3]:
triggering_results = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POS_NORMAL_CYTO']
N = 1_000

for trigger in triggering_results:
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix, p.cervical_result = 42, True, trigger

    cin_counts = Counter(
        run_colposcopy(p.__class__(**{**p.__dict__, 'cervical_result': trigger,
                                      'event_log': []}), 0)
        for _ in range(N)
    )
    expected = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{trigger}', {})
    print(f'\nColposcopy from {trigger} (n={N:,}):')
    for grade in ['NORMAL', 'CIN1', 'CIN2', 'CIN3']:
        obs  = cin_counts.get(grade, 0) / N * 100
        exp  = expected.get(grade, 0) * 100
        print(f'  {grade:<8} observed={obs:5.1f}%  expected={exp:5.1f}%')


Colposcopy from ASCUS (n=1,000):
  NORMAL   observed= 62.4%  expected= 60.0%
  CIN1     observed= 24.2%  expected= 25.0%
  CIN2     observed=  9.5%  expected= 10.0%
  CIN3     observed=  3.9%  expected=  5.0%

Colposcopy from LSIL (n=1,000):
  NORMAL   observed= 40.8%  expected= 40.0%
  CIN1     observed= 33.7%  expected= 35.0%
  CIN2     observed= 15.7%  expected= 15.0%
  CIN3     observed=  9.8%  expected= 10.0%

Colposcopy from ASC-H (n=1,000):
  NORMAL   observed= 26.0%  expected= 25.0%
  CIN1     observed= 19.2%  expected= 20.0%
  CIN2     observed= 28.8%  expected= 30.0%
  CIN3     observed= 26.0%  expected= 25.0%

Colposcopy from HSIL (n=1,000):
  NORMAL   observed= 11.5%  expected= 10.0%
  CIN1     observed=  9.3%  expected= 10.0%
  CIN2     observed= 27.0%  expected= 30.0%
  CIN3     observed= 52.2%  expected= 50.0%

Colposcopy from HPV_POS_NORMAL_CYTO (n=1,000):
  NORMAL   observed= 50.4%  expected= 50.0%
  CIN1     observed= 28.6%  expected= 30.0%
  CIN2     observed= 15.4%

## Full Cervical Follow-Up — End-to-End Trace
Run 10 patients from screening result → colposcopy → treatment/surveillance.
Print each patient's event log.

In [4]:
from screening import run_screening_step, get_eligible_screenings

random.seed(99)
DAY = 0
metrics = initialize_metrics()

n_run = 0
traced = []
pid = 0

while n_run < 10:
    p = sample_patient(pid, DAY, 'gynecologist', 'outpatient')
    pid += 1
    if 'cervical' not in get_eligible_screenings(p):
        continue

    result = run_screening_step(p, 'cervical', DAY)
    if result is None:
        continue

    disposition = run_cervical_followup(p, DAY, metrics)
    p.log(DAY, f'DISPOSITION: {disposition}')
    traced.append(p)
    n_run += 1

# Print traces
for p in traced:
    p.print_history()


── Patient 0 | age=43 | destination=gynecologist ──
  Day     0: SCREEN cervical via hpv_alone
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance
  Day     0: DISPOSITION: routine_surveillance

── Patient 1 | age=53 | destination=gynecologist ──
  Day     0: SCREEN cervical via co_test
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance
  Day     0: DISPOSITION: routine_surveillance

── Patient 2 | age=51 | destination=gynecologist ──
  Day     0: SCREEN cervical via co_test
  Day     0: RESULT cervical: ASC-H
  Day     0: ROUTE ASC-H → colposcopy
  Day     0: COLPOSCOPY → NORMAL
  Day     0: TREATMENT — NORMAL: 1-year surveillance
  Day     0: DISPOSITION: surveillance

── Patient 3 | age=28 | destination=gynecologist ──
  Day     0: SCREEN cervical via cytology
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance
  Day     0: DISPOSITION: routine_surveilla

## Disposition Summary

In [5]:
from metrics import print_summary

# Populate metrics from traced patients
metrics['n_patients'] = len(traced)
metrics['n_eligible_any'] = len(traced)
for p in traced:
    metrics['n_screened']['cervical'] += 1
    if p.cervical_result:
        metrics['cervical_results'][p.cervical_result] += 1
    if p.exit_reason:
        from metrics import record_exit
        record_exit(metrics, p.exit_reason)

print_summary(metrics)

NYP WOMEN'S HEALTH SCREENING SIMULATION — RESULTS

Patients simulated:                            10
Eligible for ≥1 screening:                     10
Unscreened (declined / no-show):                0  (0.0%)
  ↳ agreed to reschedule:                       0  (0.0% of unscreened)

Screenings completed by cancer type:
  cervical                     10

Cervical result distribution  (n=10):
  ASC-H                               1  (10.0%)
  NORMAL                              9  (90.0%)
  Abnormal rate:                 10.0%

Colposcopies performed:                  1  (100.0% of abnormals)
  CIN grade distribution:
    NORMAL            1

Treatments by type:
  surveillance                  1

Outcomes:
  Treated:                                    0  (0.0% of colposcopies)
  Untreated:                                  0
  Lost to follow-up:                          0  (0.0% of all patients)

LTFU breakdown:
  Post-abnormal screen:                       0
  Post-colposcopy:             